In [20]:
import os
from typing import TypedDict, Annotated, List
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
import operator

In [21]:
# Load environment variables
load_dotenv()

True

### STATE DEFINITION

In [22]:
# STATE DEFINITION

class ResearchState(TypedDict):
    """State object that flows through the agent graph"""
    topic: str
    research_plan: str
    search_queries: List[str]
    research_findings: Annotated[List[str], operator.add]  # Accumulate findings
    report_draft: str
    final_report: str
    iteration_count: int
    needs_refinement: bool

### AGENT NODES

In [23]:
class ResearchAgent:
    """Main research agent orchestrating the workflow"""
    
    def __init__(self, model_name: str = "gpt-4o-mini", temperature: float = 0.7):
        """Initialize the research agent with OpenAI model"""
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            api_key=os.getenv("OPENAI_API_KEY")
        )
        self.max_iterations = 2
    
    def planner_node(self, state: ResearchState) -> ResearchState:
        """
        Planning Node: Creates a research strategy and generates search queries
        """
        print("\n🎯 Planning research strategy...")
        
        prompt = f"""You are a research planning expert. Given the topic: "{state['topic']}"
        
        Create a comprehensive research plan with 3-5 specific search queries that will help gather 
        diverse and relevant information. Make queries specific and actionable.
        
        Format your response as:
        PLAN: [Brief research strategy]
        QUERIES:
        1. [Query 1]
        2. [Query 2]
        3. [Query 3]
        ...
        """
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        content = response.content
        
        # Parse the response
        plan = content.split("QUERIES:")[0].replace("PLAN:", "").strip()
        queries_text = content.split("QUERIES:")[1].strip()
        queries = [q.strip().split(". ", 1)[1] for q in queries_text.split("\n") if q.strip()]
        
        return {
            **state,
            "research_plan": plan,
            "search_queries": queries,
            "iteration_count": state.get("iteration_count", 0)
        }
    
    def researcher_node(self, state: ResearchState) -> ResearchState:
        """
        Research Node: Simulates web searches and gathers information
        In production, this would use Tavily or another search API
        """
        print("\n🔍 Gathering research data...")
        
        findings = []
        
        for query in state["search_queries"]:
            prompt = f"""As a research assistant, provide detailed information about: "{query}"
            
            Include:
            - Key facts and statistics
            - Important concepts and definitions
            - Recent developments or trends
            - Relevant examples
            
            Be comprehensive but concise (2-3 paragraphs).
            """
            
            response = self.llm.invoke([HumanMessage(content=prompt)])
            findings.append(f"Query: {query}\n{response.content}\n")
            print(f"  ✓ Researched: {query[:60]}...")
        
        return {
            **state,
            "research_findings": findings
        }
    
    def writer_node(self, state: ResearchState) -> ResearchState:
        """
        Writer Node: Synthesizes research findings into a structured report
        """
        print("\n✍️  Writing research report...")
        
        # Combine all findings
        all_findings = "\n\n".join(state["research_findings"])
        
        prompt = f"""You are an expert research writer. Create a comprehensive, well-structured report on: "{state['topic']}"

Research Plan: {state['research_plan']}

Research Findings:
{all_findings}

Write a professional report with:
1. Executive Summary
2. Introduction
3. Main Findings (organized by themes)
4. Analysis and Insights
5. Conclusion

Make it informative, well-organized, and engaging. Use markdown formatting.
"""
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        
        return {
            **state,
            "report_draft": response.content
        }
    
    def reviewer_node(self, state: ResearchState) -> ResearchState:
        """
        Reviewer Node: Reviews the report and decides if refinement is needed
        """
        print("\n👀 Reviewing report quality...")
        
        prompt = f"""You are a senior editor reviewing this research report on "{state['topic']}".

Report:
{state['report_draft']}

Evaluate the report on:
1. Completeness - Does it cover the topic thoroughly?
2. Structure - Is it well-organized?
3. Quality - Is the writing clear and professional?
4. Insights - Does it provide valuable analysis?

Respond with either:
APPROVED: [Brief comment on why it's ready]
OR
NEEDS_WORK: [Specific suggestions for improvement]
"""
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        content = response.content
        
        if "APPROVED" in content:
            print("  ✓ Report approved!")
            return {
                **state,
                "final_report": state["report_draft"],
                "needs_refinement": False
            }
        else:
            print("  ⚠ Report needs refinement")
            return {
                **state,
                "needs_refinement": True,
                "iteration_count": state["iteration_count"] + 1
            }
    
    def refiner_node(self, state: ResearchState) -> ResearchState:
        """
        Refiner Node: Improves the report based on reviewer feedback
        """
        print("\n🔧 Refining report...")
        
        prompt = f"""Improve this research report on "{state['topic']}" to make it more comprehensive and professional.

Current Report:
{state['report_draft']}

Enhance the report by:
- Adding more depth and analysis
- Improving structure and flow
- Ensuring all key points are covered
- Making it more engaging and insightful

Provide the improved version using markdown formatting.
"""
        
        response = self.llm.invoke([HumanMessage(content=prompt)])
        
        return {
            **state,
            "report_draft": response.content
        }


### GRAPH CONSTRUCTION

In [24]:
def should_refine(state: ResearchState) -> str:
    """Conditional edge: Decides whether to refine or finish"""
    if state.get("needs_refinement", False) and state["iteration_count"] < 2:
        return "refine"
    return "finish" 

In [25]:
def create_research_graph():
    """Constructs the LangGraph workflow"""
    
    # Initialize agent
    agent = ResearchAgent()
    
    # Create the graph
    workflow = StateGraph(ResearchState)
    
    # Add nodes
    workflow.add_node("planner", agent.planner_node)
    workflow.add_node("researcher", agent.researcher_node)
    workflow.add_node("writer", agent.writer_node)
    workflow.add_node("reviewer", agent.reviewer_node)
    workflow.add_node("refiner", agent.refiner_node)
    
    # Define edges (transitions)
    workflow.add_edge("planner", "researcher")
    workflow.add_edge("researcher", "writer")
    workflow.add_edge("writer", "reviewer")
    
    # Conditional edge: review → refine or end
    workflow.add_conditional_edges(
        "reviewer",
        should_refine,
        {
            "refine": "refiner",
            "finish": END
        }
    )
    
    # After refinement, go back to reviewer
    workflow.add_edge("refiner", "reviewer")
    
    # Set entry point
    workflow.set_entry_point("planner")
    
    # Compile with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)
    
    return app


### EXECUTION


In [27]:
def run_research_assistant(topic: str):
    """
    Main execution function
    
    Args:
        topic: The research topic to investigate
    """
    print("=" * 80)
    print("🤖 MULTI-AGENT RESEARCH ASSISTANT")
    print("=" * 80)
    print(f"\n📚 Research Topic: {topic}\n")
    
    # Create the graph
    app = create_research_graph()
    
    # Initial state
    initial_state = {
        "topic": topic,
        "research_plan": "",
        "search_queries": [],
        "research_findings": [],
        "report_draft": "",
        "final_report": "",
        "iteration_count": 0,
        "needs_refinement": False
    }
    
    # Run the agent workflow
    config = {"configurable": {"thread_id": "research_session_1"}}
    
    try:
        # Execute the graph
        final_state = None
        for state in app.stream(initial_state, config):
            final_state = state
        
        # Extract final report
        if final_state and "reviewer" in final_state:
            final_report = final_state["reviewer"].get("final_report", "")
            
            print("\n" + "=" * 80)
            print("📄 FINAL RESEARCH REPORT")
            print("=" * 80)
            print(final_report)
            
            # Save to file
            filename = f"research_report_{topic[:30].replace(' ', '_')}.md"
            with open(filename, "w") as f:
                f.write(f"# Research Report: {topic}\n\n")
                f.write(final_report)
            print(f"\n✅ Report saved to: {filename}")
            
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        raise


### MAIN ENTRY POINT


In [28]:
if __name__ == "__main__":
    # Example usage
    research_topic = "The impact of artificial intelligence on software development practices"
    
    # Ensure API key is set
    if not os.getenv("OPENAI_API_KEY"):
        print("⚠️  Please set OPENAI_API_KEY environment variable")
        print("   You can create a .env file with: OPENAI_API_KEY=your_key_here")
        exit(1)
    
    run_research_assistant(research_topic)
    
    print("\n" + "=" * 80)
    print("✨ Research complete!")
    print("=" * 80)

🤖 MULTI-AGENT RESEARCH ASSISTANT

📚 Research Topic: The impact of artificial intelligence on software development practices


🎯 Planning research strategy...

🔍 Gathering research data...
  ✓ Researched: "impact of artificial intelligence on software coding practi...
  ✓ Researched: "AI tools for software testing and their effectiveness"...
  ✓ Researched: "case studies on AI integration in software development team...
  ✓ Researched: "how AI is changing project management in software developme...
  ✓ Researched: "future trends of AI in software engineering practices"...

✍️  Writing research report...

👀 Reviewing report quality...
  ✓ Report approved!

📄 FINAL RESEARCH REPORT
# The Impact of Artificial Intelligence on Software Development Practices

## Executive Summary

The integration of Artificial Intelligence (AI) into software development practices has dramatically reshaped the landscape of coding, testing, project management, and overall development workflows. This report delve